In [0]:
from pyspark.sql import functions as F

In [0]:
# lets specify start date and end date
start_date = "2024-01-01"
end_date = "2025-12-01"

In [0]:
# now create one row per month which will be the start date of each month
df = (
    spark.sql(f"""
        SELECT explode(
            sequence (
                to_date('{start_date}'),
                to_date('{end_date}'),
                interval 1 month
            )
        ) AS month_start_date
    """)
)

# explode() is a table-generating function (UDTF) in Spark. It takes an array (or a map) from a single column and unrolls it vertically, producing a new row for each element in that collection.


# Now you have all the start dates as rows lets add some columns for analytics
df = (
    df
    
    .withColumn("date_key", F.date_format("month_start_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("month_start_date"))
    .withColumn("month_name", F.date_format("month_start_date", "MMMM"))
    .withColumn("month_short_name", F.date_format("month_start_date", "MMM"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("month_start_date")))
    .withColumn("year_quarter", F.concat(F.col("year"), F.lit("-Q"), F.quarter("month_start_date")))
)

In [0]:
display(df)

In [0]:
## Save the table
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("fmcg.gold.dim_date")